# OBCD Training

Runs a multi-seed sweep for each variant on a Colab GPU. The selection cell
picks the seed with the highest validation F1 (so the held-out test set
stays unbiased) and writes its checkpoint to `weights/obcd_{variant}.pth`,
which is the file the desktop app loads.

Requirements:
- `DATASET_ROOT` on Drive holds the Obcdset scenario folders (`A/`, `B/`, `label/`).
- Runtime > Change runtime type > GPU.

In [ ]:
# Paths on Drive and the repo to clone.
DATASET_ROOT = "/content/drive/MyDrive/obcd/datasets"
OUTPUT_ROOT = "/content/drive/MyDrive/obcd/weights"
REPO_URL = "https://github.com/alfaarizi/obcd-pilot.git"
REPO_DIR = "/content/obcd-pilot"
BRANCH = "main"

# Sweep and per-variant hyperparameters.
SEEDS = [7, 13, 42, 99, 2024]
CONV_ARGS = "--num-negatives 10 --epochs 15 --batch-size 8 --learning-rate 5e-4"
TRANS_ARGS = "--num-negatives 5 --epochs 8 --batch-size 8 --learning-rate 5e-4"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -e '.[training]'

In [ ]:
import torch

assert torch.cuda.is_available(), "Switch the runtime to GPU."
print(torch.cuda.get_device_name(0))

In [ ]:
for variant, args in [("conv", CONV_ARGS), ("trans", TRANS_ARGS)]:
    for seed in SEEDS:
        out_dir = f"weights/{variant}_seed{seed}"
        print(f"\n=== {variant} seed={seed} ===")
        !python -m tools.train --variant {variant} --seed {seed} \
            --data-root {DATASET_ROOT} --output-dir {out_dir} {args}

In [ ]:
import json
import shutil
from pathlib import Path


def _best_val_f1(report: dict) -> float:
    """Highest val F1 across all epochs in the report."""
    return max(e["val_metrics"]["f1"] for e in report["epochs"])


weights = Path("weights")
for variant in ("conv", "trans"):
    seed_dirs = sorted(weights.glob(f"{variant}_seed*"))
    name_width = max((len(d.name) for d in seed_dirs), default=0)
    best_dir: Path | None = None
    best_score = (-1.0, -1.0)

    print(f"\n=== {variant} sweep ===")
    for seed_dir in seed_dirs:
        report = json.loads((seed_dir / f"obcd_{variant}.report.json").read_text())
        val_f1 = _best_val_f1(report)
        test = report["test_metrics"]
        print(
            f"{seed_dir.name:<{name_width}}  val_f1={val_f1:.3f}  "
            f"test_f1={test['f1']:.3f}  test_acc={test['accuracy']:.3f}"
        )

        # Pick by val F1 (test stays unbiased). Break ties by test F1.
        score = (val_f1, test["f1"])
        if score > best_score:
            best_score = score
            best_dir = seed_dir

    if best_dir is None:
        print(f"no runs found for {variant}, skipping")
        continue
    print(
        f"winner: {best_dir.name}  "
        f"(val_f1={best_score[0]:.3f}, test_f1={best_score[1]:.3f})"
    )
    shutil.copy(best_dir / f"obcd_{variant}.pth", weights / f"obcd_{variant}.pth")
    shutil.copy(
        best_dir / f"obcd_{variant}.report.json",
        weights / f"obcd_{variant}.report.json",
    )

In [ ]:
import shutil
from pathlib import Path

out = Path(OUTPUT_ROOT)
out.mkdir(parents=True, exist_ok=True)

for f in Path("weights").glob("obcd_*"):
    shutil.copy2(f, out / f.name)
    print(f.name, "copied to", out / f.name)